# Spiking Connectome × Evolution Strategy (Colab GPU)

End-to-end **brain-style** experiment, no backprop / no critic:
- **structure** = real larva connectome (389-node graph from 2 CSVs)
- **neuron** = leaky integrate-and-fire with soft spikes
- **learning** = Evolution Strategy (natural-selection-style; the environment is the critic)

Everything is GPU-batched: `pop_size` candidates × `episodes_per_cand` episodes are
rolled out simultaneously in a vectorized torch OSL env. **No Drive mount needed** —
the repo is cloned from GitHub.

## 1. Clone repo + GPU check (no Drive mount)

In [ ]:
import os, sys, subprocess, torch
REPO_URL = 'https://github.com/InHyunseo/Brain-inspired-OSL.git'
REPO_DIR = '/content/2d-osl'
if not os.path.isdir(REPO_DIR):
    subprocess.check_call(['git','clone','--depth','1', REPO_URL, REPO_DIR])
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path: sys.path.insert(0, REPO_DIR)
# assets present?
for p in ('assets/connectome/weights.csv','assets/connectome/metadata.csv',
          'src/brain/es_gpu.py'):
    assert os.path.exists(p), f'MISSING {p} — is src/brain committed & pushed?'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('repo:', REPO_DIR)
print('device:', DEVICE, '|', torch.cuda.get_device_name(0) if DEVICE=='cuda' else 'CPU only — enable GPU runtime')

> If a cell above fails on `src/brain/...`, the brain code hasn't been pushed to the
> repo yet. Commit & push `src/brain/` first, or upload those files manually.

## 2. Config

In [ ]:
from src.brain.es_gpu import ESGpu, ESGpuConfig

cfg = ESGpuConfig(
    pop_size          = 256,    # candidates per generation (even). bigger = better gradient, more VRAM
    episodes_per_cand = 8,      # parallel episodes averaged per candidate
    sigma             = 0.05,   # perturbation std
    lr                = 0.03,   # ES step size
    weight_decay      = 0.005,
    max_steps         = 600,    # env steps per episode
    inner_steps       = 4,      # message-passing steps per env step
    n_output_nodes    = 8,
    success_bonus     = 50.0,
    seed              = 0,
)
GENERATIONS = 300

es = ESGpu(cfg, device=DEVICE)
print('connectome params:', es.n_params, '| total parallel rollouts/gen:', cfg.pop_size*cfg.episodes_per_cand)

## 3. Train

Each generation = one big batched GPU rollout of all `pop_size × episodes_per_cand`
episodes. Watch `success` — that's the fraction of episodes reaching the source.

In [ ]:
hist = es.train(generations=GENERATIONS, log_every=5)

## 4. Plot training curves

In [ ]:
import matplotlib.pyplot as plt
g  = [h['generation'] for h in es.history]
fm = [h['fit_mean'] for h in es.history]
fx = [h['fit_max'] for h in es.history]
sm = [h['success_mean']*100 for h in es.history]
fig, ax = plt.subplots(1,2, figsize=(12,4))
ax[0].plot(g, fm, label='fitness mean'); ax[0].plot(g, fx, label='fitness max', alpha=.7)
ax[0].set_xlabel('generation'); ax[0].set_ylabel('fitness (env return)'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[0].set_title('ES fitness over generations')
ax[1].plot(g, sm, color='tab:green'); ax[1].set_xlabel('generation'); ax[1].set_ylabel('success rate (%)')
ax[1].set_title('Success rate over generations'); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 5. Save the evolved brain

In [ ]:
import numpy as np, json, time
out = f'/content/es_connectome_gen{es.generation}.npz'
np.savez(out, theta=es.theta, history=json.dumps(es.history),
         config=json.dumps(cfg.__dict__))
print('saved', out)
# download to local machine:
try:
    from google.colab import files; files.download(out)
except Exception as e:
    print('(manual download)', e)

## 6. (optional) Tuning notes

- **No learning?** Try larger `sigma` (0.08–0.1) early, or raise `input_gain` in
  `ESGpuConfig` so the LIF neurons actually fire (initial spike rate is low).
- **Too slow / OOM?** Lower `pop_size` or `episodes_per_cand`; raise them if VRAM allows.
- This env is **clean-field only** (no bump noise) — it's the easy regime to first
  prove the spiking connectome can learn at all with ES.